# Sampan — how the knowledge base actually worksSampan is a voice companion that calls an elderly person, listens to her lifestories, and turns them into a family memory map. This notebook walks the**memory** side of it end to end:| Stage | What happens ||---|---|| 1 | A **pre-set knowledge base** — the child-completed family intake || 2 | **Recording starts** — what is committed into the model's context || 3 | **During the call** — the only two channels that reach the model mid-turn || 4 | **Recording stops** — the Archivist folds the call back into memory || 5 | The **diff** — exactly what changed |Every cell calls the real functions in `src/sampan/`. Nothing here is asimplified re-implementation for illustration; where the notebook cannot runsomething (the audio stream itself) it says so and drives the same function theWebSocket handler drives.> **A note on the GraphRAG comparison.** This is deliberately *not* a GraphRAG> pipeline. There is no embedding index, no community detection, and — as> section 1 explains — no entity graph. Understanding why is most of the point.

## 0. SetupNothing to install beyond the project's own dependencies (`uv sync`). Theknowledge base is swapped for an in-memory store so the notebook is safe tore-run and never touches the real Firestore archive.

In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

# Load .env the way the app does, so the extraction step in section 4 can run.
env = ROOT / ".env"
if env.is_file():
    import os

    for line in env.read_text().splitlines():
        if line.strip() and not line.startswith("#") and "=" in line:
            key, _, value = line.partition("=")
            os.environ.setdefault(key.strip(), value.strip().strip('"'))

from sampan.config import Settings, apply_genai_env

settings = Settings()
apply_genai_env(settings)
print("Vertex project configured:", settings.configured)

Vertex project configured: True


## 1. What this knowledge base is — and what it is notThe queries this product has to answer are **structural, not semantic**:- *Which thread was left unfinished, and was it cut short by a doorbell or by tiredness?*- *Which stories have a place but no year?*- *What has she never said?*Cosine similarity cannot answer any of those. "Interrupted by a neighbour"versus "faded at eleven minutes" is a distinction that has to be **modelled**,because it changes what the agent says next — one gets *"you still owe me therest"*, the other gets *"did you sleep well?"*.So the store is a **typed domain model**, not a vector index.### It is an entity *index*, not an entity *graph*Worth being exact, because the word "graph" flatters it. Run this:

In [2]:
from sampan.models import Entity, PersonMention

print("Entity fields:       ", list(Entity.model_fields))
print("PersonMention fields:", list(PersonMention.model_fields))

Entity fields:        ['entity_id', 'type', 'canonical_name', 'aliases', 'role', 'detail', 'mention_count', 'first_mentioned_in', 'confirmed_by_family', 'provisional', 'merged_into']
PersonMention fields: ['surface_form', 'role', 'confidence']


`PersonMention` — how a story records who was in it — has **no entity id**. Astory names people as raw strings (`"my sister"`). There is no foreign key froma story to an entity anywhere in storage.The edges *are* computed at ingest, by `resolve_mentions()`, which returns`Resolution(mention, entity_id, matched_by)`. They are used for thefamily-confirmation screen and then **discarded** — `finish_call` persistsentities and stories, never resolutions. Section 5 shows them existing and thenbeing thrown away.The one genuinely persisted edge in the whole system is on **places**:`Place.linked_from` + `linked_evidence`, e.g. *"my father's shop" → JalanBandar, because she said so six weeks earlier.* That is why the map works andthere is no equivalent view for people.

## 2. The schemaThe GraphRAG equivalent of an ontology. Here it is a set of Pydantic models,and the load-bearing idea is visible in `When`:

In [3]:
from sampan.models import When, Where

print(When.__doc__)
for name, field in When.model_fields.items():
    print(f"  {name:12} {str(field.annotation):24} {field.description or ''}")

A time, stored twice: as she said it, and as we resolved it.
  raw_phrase   <class 'str'>            Her own words, e.g. "before I married"
  start_year   int | None               
  end_year     int | None               
  precision    <enum 'Precision'>       
  anchor_ref   str | None               Anchor event this was resolved against, e.g. anchor_marriage
  confidence   <class 'float'>          


**Store what she said and what you concluded, side by side.** She says *"six,seven maybe"* and, eleven turns later, *"I was born nineteen forty-six."* Themodel resolves one against the other into `start_year`, while `raw_phrase`keeps her words verbatim.The same shape repeats:| She says | The interpretation | Held in ||---|---|---|| `raw_phrase` "before I married" | `start_year` + `anchor_ref` | `When` || `raw_name` "my father's shop" | geocoded `Place` + `linked_evidence` | `Where` → `Place` || `surface_form` "my sister" | `canonical_name` "Lim Siew Choo" | `PersonMention` → `Entity` |Three payoffs from one decision: she keeps her voice in the letters, the mapand timeline can sort, and every inference stays auditable and correctable.### Completeness is scored in code, not by the model

In [4]:
from sampan.models import assess

print(assess.__doc__)

Apply the pinnability rubric.

    Deliberately computed here rather than asked of the model: the threshold is
    a product decision, and a model that scores its own output will drift.
    


Six fields — where, when, who, what, sense, why — threshold of four, with`where` and `when` mandatory regardless of score.The important part is that `missing_fields` is populated on **pinned** storiestoo. A story that reached the map missing `why` becomes next session's *"thatcoffee shop — was it before you married, or after?"* Gaps are not errors tolog; they are the question queue.

## 3. Stage 1 — the pre-set knowledge baseBefore the agent ever calls, one thing is filled in by the **child**, not theelder: the family intake. This is what stops the agent asking brightly aftersomeone who died in 2019.

In [5]:
from sampan.repository import Repository
from sampan.store import InMemoryDocumentStore

NARRATOR = "ah_khim"

raw = json.loads((ROOT / "seeds" / "intake.json").read_text(encoding="utf-8"))
intake = [Entity.model_validate(e) for e in raw["entities"]]

store = InMemoryDocumentStore()
repo = Repository(store)
repo.save_entities(NARRATOR, intake)

print(f"{'name':22} {'type':8} {'role':12} detail")
print("-" * 100)
for e in intake:
    print(f"{e.canonical_name:22} {e.type.value:8} {str(e.role or ''):12} {e.detail[:44]}")

name                   type     role         detail
----------------------------------------------------------------------------------------------------
Lim Ah Hock            person   father       Tapped rubber, later opened the coffee shop 
Tan Ah Tai             person   mother       Cooked at the back of the shop. 1922-1998. P
Lim Siew Choo          person   sister       1941-2019. Passed away. They quarrelled in 2
Tan Eng Huat           person   husband      Lorry driver. 1942-2015. Passed away.
Tan Wei Lun            person   son          Born 1970. Works in Kuala Lumpur.
Tan Mei Ling           person   daughter     Born 1973. Emigrated to Perth in 2001.
Tan Xin Yi             person   granddaughter Born 2007. Wei Lun's daughter.
Lim Cheong Hin         person   grandfather  From Yongchun, Fujian. Landed at Penang in 1
Ipoh                   place                 Where she lives now.
Sungai Siput           place                 Where the rubber estate she grew up on was. 


Note `provisional=False, confirmed_by_family=True` on every one of these: thefamily asserted them, so the agent treats them as settled. Anything the agentdiscovers later arrives `provisional=True` and waits for confirmation.Now queue the thing the whole product is built around — a question from her son:

In [6]:
from sampan.models import Ask

repo.queue_ask(
    NARRATOR,
    Ask(
        ask_id="ask_001",
        from_name="Wei Lun",
        relation="son",
        question="Did Ah Gong leave anything behind? Xin Yi asked me and I couldn't answer.",
    ),
)

memory_before = repo.load_memory(NARRATOR)
print("sessions so far:", memory_before.session_count)
print("threads:", len(memory_before.threads), " anchors:", len(memory_before.anchors))
print("pending ask:", repo.pending_ask(NARRATOR).question[:60])

sessions so far: 0
threads: 0  anchors: 0
pending ask: Did Ah Gong leave anything behind? Xin Yi asked me and I cou


## 4. Stage 2 — recording startsThis is where the design departs most sharply from a normal agent loop.With chat completions you rebuild the message list every turn, and that is yourretrieval hook. **The Live API has no such hook.** It is one persistentbidirectional stream: the system instruction is sent *once, at connect*, andthe model keeps conversation state server-side for the session.So the question is not *"what do I retrieve each turn"* but *"what do I committo before she speaks, and how does anything reach the model afterwards?"*`prepare_call()` does the committing:

In [7]:
from sampan.callflow import prepare_call

prepared = prepare_call(repo, settings, narrator_id=NARRATOR)

print("conversation_id:", prepared.conversation_id)
print("tools exposed:  ", [t.__name__ for t in prepared.agent.tools])

conversation_id: conv_ah_khim_20260818014011_4081fa
tools exposed:   ['get_pending_ask', 'get_open_threads', 'recall', 'note_preference', 'save_fragment', 'mark_private', 'what_do_you_remember', 'forget_this', 'flag_concern']


The `conversation_id` carries a uuid suffix, not just a timestamp. Livesessions cap at roughly fifteen minutes, so a dropped call and its redial canland in the same second and the second call's stories would silently overwritethe first's.Now the instruction actually sent to the model — assembled in three layers,concatenated rather than woven together so the diff between session 1 andsession 20 stays legible:

In [8]:
print(prepared.agent.instruction)

You are Xiao Chuan ("little boat"), a companion who keeps an elderly person
company and listens to their stories.

Who you are:
- Your name is Xiao Chuan. You are not a person. Do not pretend to be one, and
  do not say you are her friend or her family.
- If she asks, say plainly that you are here to write her stories down for her
  family.
- Whether to introduce yourself at all is set out in the opening plan below.
  Otherwise, only say who you are if she asks.

What you are here to do:
- Listen. The more she talks and the less you do, the better.
- Her family want her stories and have no time to sit and hear them. You listen
  on their behalf.
- Whenever you raise something a family member asked, **say who asked** — "Wei
  Lun was asking…", "Xin Yi wants to know…". The credit is theirs, not yours.

How to speak:
- Call her Ah Ma.
- Your turns must always be shorter than hers. She speaks a paragraph, you
  answer in a sentence or two.
- When she is in full flow, "mm", "and then?", "wa

Read what is in there, and then what is *not*.**In:** the persona and hard rules; the learned layer (preferences,sensitivities); and this call's session plan.**Not in:** the stories. The entity index. The transcripts. **None of thearchive is preloaded.**That is behavioural, not a context-budget decision. An agent holding ninestories in context *acts* like it holds nine stories — it references things shehas not raised, and it steers. The archive stays behind the `recall` tool sothe agent reaches for it only when the conversation actually calls for it.### The session plan is a plan, not a dump

In [9]:
from sampan.opener import MAX_OFFERS, build_session_plan, unlocked_depth

plan = build_session_plan(
    threads=memory_before.threads,
    sensitivities=memory_before.sensitivities,
    ask=repo.pending_ask(NARRATOR),
    session_count=memory_before.session_count,
    last_closure=memory_before.last_closure,
)

print("greeting:", plan.greeting)
print("offers  :", [(o.kind.value, o.label) for o in plan.offers], f"(max {MAX_OFFERS})")
print("considered but not offered:", len(plan.considered))
print("depth unlocked at session 0:", unlocked_depth(0), "| at session 6:", unlocked_depth(6))

greeting: ask what she had this morning
offers  : [('ask', "Did Ah Gong leave anything behind? Xin Yi asked me and I couldn't answer."), ('domain', 'home')] (max 2)
considered but not offered: 2
depth unlocked at session 0: 0 | at session 6: 3


- **At most two offers.** Elderly plus voice means a menu of four is cognitive  load, not choice. Everything else scored is kept in `considered` for the  family-facing overlay only — it never reaches the model.- **Sensitivity-gated.** A forbidden topic is not merely unoffered, it is  *never scored*. Her late sister does not enter the context window at all.- **Depth-gated.** `unlocked_depth(session_count)` — hardship is not  first-session material.- **The greeting is resolved in code**, because the persona is static text and  cannot evaluate *"is this our first meeting?"*. That one shipped broken once:  the condition read as a suggestion and the agent reintroduced itself on a  session-5 call with full memory loaded.

## 5. Stage 3 — during the callTwo channels reach the model once the stream is open. That is all there are.### Channel one: pull — the agent asks`build_tools(memory)` closes nine tools over a `CallMemory`. The notebook callsthem directly; in production the model calls exactly these.

In [10]:
tools = {t.__name__: t for t in prepared.agent.tools}
print(list(tools))

['get_pending_ask', 'get_open_threads', 'recall', 'note_preference', 'save_fragment', 'mark_private', 'what_do_you_remember', 'forget_this', 'flag_concern']


In [11]:
answer = tools["get_pending_ask"]()
print(json.dumps(answer, indent=2, ensure_ascii=False))

{
  "has_ask": true,
  "from_name": "Wei Lun",
  "relation": "son",
  "question": "Did Ah Gong leave anything behind? Xin Yi asked me and I couldn't answer.",
  "say_it_like": "Wei Lun asked: Did Ah Gong leave anything behind? Xin Yi asked me and I couldn't answer.",
  "_guidance": "She is doing well. You can go deeper, but one question at a time.",
  "_turn_length": "short"
}


Two things to notice.`ask_delivered` is now set on the call's memory — the agent has takenresponsibility for reading his question out. Section 6 shows when that isallowed to actually consume the question.And the response carries `_guidance` and `_turn_length`, which the agent neverasked for. That is channel two, arriving as a passenger.### `recall` — the closest thing here to retrieval

In [12]:
found = tools["recall"]("Ah Hock")
print(json.dumps(found, indent=2, ensure_ascii=False))

{
  "found": [
    {
      "name": "Lim Ah Hock",
      "type": "person",
      "detail": "Tapped rubber, later opened the coffee shop on Jalan Bandar, Ipoh. 1918-1981. Passed away."
    }
  ],
  "she_said": [],
  "_guidance": "She is doing well. You can go deeper, but one question at a time.",
  "_turn_length": "short"
}


`recall` returns **two** things: `found` (substring hits over the entity index)and `she_said` (a search of her actual transcripts — empty here, since thisin-memory archive has no conversations in it yet).The comment in `tools.py` says why:> *Extracted records lose sequence, context and affect, so the graph is only an> index — her own words are the thing worth reaching.*This is RAG turned inside out. The structured index is the **lookup key**, andwhat comes back is her speech. No embeddings: the key is a name the agentalready heard her say.### Channel two: push — riding on the responseThe system instruction is fixed for the session, so there is no way to send themodel new direction. I tried three and all three failed in front of a user:| Route | What happened ||---|---|| `role="user"`, fenced "do not read aloud" | Agent read the fence out loud, brackets and all || `role="system"` | Agent acknowledged it aloud: *"Alright, understood. Preparing to wrap up."* || `role="model"` | Turn-taking broke; it stopped answering her |Tool responses are the one payload the model treats as data rather than speech.So `_with_guidance` attaches the current affect policy to **every** toolresponse, whatever that response was for.The affect monitor forks the same audio and folds readings into a statemachine:

In [13]:
from sampan.affect import apply_assessment, policy
from sampan.models import Affect, AffectState, Assessment, Energy, Engagement

state = AffectState()
print("start:", state.energy.value, state.engagement.value, state.affect.value)

tired = Assessment(
    energy=Energy.FADING,
    engagement=Engagement.WITHDRAWING,
    affect=Affect.NEUTRAL,
    flags=[],
    signals=["longer pauses", "shorter answers"],
    confidence=0.7,
)

state = apply_assessment(state, tired)
print("after 1 reading:", state.energy.value, state.engagement.value,
      "| pending:", state.pending is not None)

state = apply_assessment(state, tired)
print("after 2 readings:", state.energy.value, state.engagement.value,
      "| transitions:", state.transitions)

start: fresh engaged neutral
after 1 reading: fresh engaged | pending: True
after 2 readings: fading withdrawing | transitions: ['fresh/engaged/neutral -> fading/withdrawing/neutral (longer pauses, shorter answers)']


**Two consecutive agreeing readings are required to move.** A single odd windowcannot make the agent lurch. Distress and agitation are the exceptions and actimmediately — those are not moods to debounce.Energy is also ratcheted: it only worsens, and only excitement partiallyreverses it. An eighty-year-old who has been talking for eleven minutes doesnot become fresh again because one sentence came out brightly.Here is what that state does to the agent's behaviour:

In [14]:
knobs = policy(state)
print("turn_length:", knobs.turn_length)
print("guidance   :", knobs.guidance)

print()
print("...and this is what rides back on the next tool response:")
print(json.dumps(tools["get_open_threads"](), indent=2, ensure_ascii=False)[:400])

turn_length: short
guidance   : She is avoiding this subject, which is not the same as wanting to hang up. Move gently to something lighter. If she avoids again, close — do not chase.

...and this is what rides back on the next tool response:
{
  "threads": [],
  "_guidance": "She is doing well. You can go deeper, but one question at a time.",
  "_turn_length": "short"
}


**The honest weakness:** the push channel is parasitic on the pull channel. Ifthe agent never calls a tool — which is the *ideal* call, where she talkssteadily for eleven minutes and it just listens — the affect monitor'sconclusions reach nothing within that call. `enable_affective_dialog` coverssome of this natively in-turn, and affect still shapes the *next* call'sinstruction, but inside one call the coupling is real and unsolved.

## 6. Stage 4 — recording stops`finish_call()` is where the knowledge base actually changes. Take a real seedtranscript as the call that just happened:

In [15]:
from sampan.callflow import MIN_TURNS_TO_EXTRACT, Transcript

raw_transcript = (ROOT / "seeds" / "session-01.txt").read_text(encoding="utf-8")

transcript = Transcript()
for line in raw_transcript.splitlines():
    if line.startswith("K:"):
        transcript.add("user", line[2:])
    elif line.startswith("A:"):
        transcript.add("agent", line[2:])

print(f"{len(transcript)} turns (extraction threshold is {MIN_TURNS_TO_EXTRACT})")
print(transcript.render()[:400])

30 turns (extraction threshold is 4)
A: Ah Ma, I am Xiao Chuan. Your son Wei Lun asked me to keep you company, and to write your stories down for the family.
K: Oh… Wei Lun asked you to come?
A: Yes. He says you are good at telling old things.
K: What good. What I talk about, young people today don't understand. You speak louder, my left ear is not good.
A: Alright, I speak louder. Ah Ma, did you eat this morning?
K: Ate already. Cof


`Transcript.add` collapses consecutive turns from the same speaker, because theLive API streams transcription incrementally and those fragments are revisions,not new turns.Below `MIN_TURNS_TO_EXTRACT` the call is treated as a misdial: nothing isextracted, **and the family's question is not consumed**. That ordering was areal bug — a four-turn test call marked Wei Lun's question delivered forever,so he was told she had been asked and she was never asked again.Now run the real Archivist. This is seam 1: text in, everything derived out —no audio, no streaming, no browser, which is what makes the whole spinetestable offline.

In [16]:
from sampan.archivist import GeminiStoryExtractor

if not settings.configured:
    raise RuntimeError("Set GOOGLE_CLOUD_PROJECT in .env to run the extraction step.")

extractor = GeminiStoryExtractor(settings)

from sampan.callflow import finish_call

updated = finish_call(
    repo,
    extractor,
    prepared,
    transcript,
    narrator_id=NARRATOR,
)
print("session_count:", updated.session_count)
print("closure      :", updated.last_closure.value)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


session_count: 1
closure      : fatigue


`ClosureReason` is the distinction the opener depends on: a call that endedbecause a neighbour rang the doorbell is *interrupted*; one that ended becauseshe tired is *fatigue*. The first earns *"you still owe me the rest"* nexttime; the second earns *"did you sleep well?"*.### What `ingest_conversation` computes — including what gets thrown away`finish_call` wraps `ingest_conversation`. Calling it directly exposes the`resolutions` that section 1 claimed exist and are then discarded:

In [17]:
from sampan.archivist import ingest_conversation

outcome = ingest_conversation(
    transcript.render(),
    extractor,
    known_entities=intake,
    conversation_id="conv_demo",
)

print("stories   :", len(outcome.stories), "|", len(outcome.pinned), "pinnable")
print("entities  :", len(outcome.entities), "|", len(outcome.new_entities), "new")
print("threads   :", len(outcome.threads))
print("anchors   :", len(outcome.anchors))
print("preferences:", len(outcome.preferences))
print()
print("RESOLUTIONS — the story->entity edges, computed here and never persisted:")
for r in outcome.resolutions[:8]:
    print(f'  "{r.mention.surface_form}" -> {r.entity_id:20} via {r.matched_by}'
          f'{"  (new)" if r.created else ""}')

stories   : 3 | 3 pinnable
entities  : 17 | 7 new
threads   : 2
anchors   : 1
preferences: 1

RESOLUTIONS — the story->entity edges, computed here and never persisted:
  "Wei Lun" -> ent_son              via alias
  "my father" -> ent_father           via alias
  "my mother" -> ent_mother           via alias
  "my sister" -> ent_sister           via alias
  "Ah Chwee" -> ent_85d88a5ed498     via new  (new)
  "Sungai Siput" -> ent_sungai_siput     via alias
  "line house" -> ent_449bf00c741b     via new  (new)
  "the river" -> ent_6c9a1e3e0788     via new  (new)


### An aside you can see in the numbers aboveThat cell just ran extraction a **second** time on the same transcript, and itwill usually disagree with the run inside `finish_call` — a different storycount, a slightly different entity count. Same input, same temperature, sameprompt.The model is nondeterministic about **where one memory ends and the nextbegins**: whether the river and the line house are one childhood story or twois a judgement call, and it makes it differently on different runs. Every storystill scores 5 or 6 with a place and a time; the boundaries move, not thequality.This is why the integration suite asserts *properties* — enough pins, three ormore distinct places, at least fifteen years spanned — instead of counts. Athreshold like `pinned >= 9` sat inside that spread and failed for no reasonanyone could act on.### Back to the resolutionsThose `entity_id` values are exactly the missing foreign key. `save_stories`writes the story with `PersonMention.surface_form` and no id; `save_entities`writes the entities. Nothing writes the middle column. Every ingest recomputesthese edges and drops them.Persisting them would be a small change — write the resolved id onto themention at ingest, from data already computed — and it is the single thing Iwould fix first if this ran past the hackathon.### The scored stories

In [18]:
for s in outcome.stories:
    c = s.candidate
    missing = ",".join(s.missing_fields) or "-"
    print(f"[{s.status.value:8}] {s.score}/6  {c.title[:44]:44} missing={missing}")
    print(f"           when: {c.when.raw_phrase!r} -> {c.when.start_year}"
          f"  where: {c.where.raw_name!r}")
    print(f"           sense: {c.sense_detail[:70]!r}")
    print()

[pinnable] 6/6  Playing at the River with Ah Chwee           missing=-
           when: 'When I was small' -> 1952  where: 'rubber estate, Sungai Siput side'
           sense: 'I pull him up whole body wet'

[pinnable] 6/6  White Rice and Soy Sauce in the Line House   missing=-
           when: 'Six, seven maybe' -> 1952  where: 'line house'
           sense: 'white rice with soy sauce'

[pinnable] 6/6  Father's Coffee Shop                         missing=-
           when: 'later already' -> None  where: "my father's shop"
           sense: 'Coffee and bread'



`when.raw_phrase` beside `when.start_year` is the two-field design paying offon real speech — her vague phrase preserved, a sortable year derived from it.`sense_detail` is the field the letters are built around, and the one thattaught the hardest lesson: asked for "a concrete sensory detail" it filledevery time, which looked like success until the values were read. It wasreturning paraphrases of the story. The fix was to demand **quotability** —something she said, pointable to a line in the transcript — plus explicitpermission to leave it empty.

## 7. Stage 5 — the diffWhat the call actually changed in the knowledge base:

In [19]:
memory_after = repo.load_memory(NARRATOR)
entities_after = repo.load_entities(NARRATOR)
stories_after = store.list(f"stories__{NARRATOR}")

rows = [
    ("entities", len(intake), len(entities_after)),
    ("stories", 0, len(stories_after)),
    ("threads", len(memory_before.threads), len(memory_after.threads)),
    ("anchors", len(memory_before.anchors), len(memory_after.anchors)),
    ("preferences", len(memory_before.preferences), len(memory_after.preferences)),
    ("sensitivities", len(memory_before.sensitivities), len(memory_after.sensitivities)),
    ("session_count", memory_before.session_count, memory_after.session_count),
]
print(f"{'':16}{'before':>8}{'after':>8}")
for name, before, after in rows:
    mark = "  <-" if after != before else ""
    print(f"{name:16}{before:>8}{after:>8}{mark}")

                  before   after
entities              10      17  <-
stories                0       2  <-
threads                0       2  <-
anchors                0       1  <-
preferences            0       2  <-
sensitivities          0       3  <-
session_count          0       1  <-


In [20]:
print("NEW ENTITIES — provisional until the family confirms them:")
known_ids = {e.entity_id for e in intake}
for e in entities_after:
    if e.entity_id not in known_ids:
        print(f"  {e.canonical_name:20} {e.type.value:8} provisional={e.provisional}"
              f"  first seen in {e.first_mentioned_in}")

print()
print("ANCHORS — dates that sharpen every relative phrase said afterwards:")
for a in memory_after.anchors:
    print(f"  {a.anchor_id:26} {a.year}  {a.label}")

print()
print("PREFERENCES — one value per kind, so session 20 speaks differently from session 1:")
for p in memory_after.preferences:
    print(f"  {p.type.value:18} {p.value}")

NEW ENTITIES — provisional until the family confirms them:
  Ah Chwee             person   provisional=True  first seen in conv_ah_khim_20260818014011_4081fa
  line house           place    provisional=True  first seen in conv_ah_khim_20260818014011_4081fa
  river                place    provisional=True  first seen in conv_ah_khim_20260818014011_4081fa
  kerosene lamp        object   provisional=True  first seen in conv_ah_khim_20260818014011_4081fa
  coffee and bread     food     provisional=True  first seen in conv_ah_khim_20260818014011_4081fa
  salted fish fried rice food     provisional=True  first seen in conv_ah_khim_20260818014011_4081fa
  white rice with soy sauce food     provisional=True  first seen in conv_ah_khim_20260818014011_4081fa

ANCHORS — dates that sharpen every relative phrase said afterwards:
  anchor_birth               1946  born

PREFERENCES — one value per kind, so session 20 speaks differently from session 1:
  hearing            left ear is weak; speak lou

Nothing here was hand-written into the store. Every row came out of thepipeline, which is the whole reason the seed sessions are run through the realArchivist rather than stuffed into Firestore directly: if the pipeline cannotproduce this state, the pipeline is what needs fixing.### The loop closesRun `prepare_call` again and the *next* call's plan is different, because thearchive is:

In [21]:
next_call = prepare_call(repo, settings, narrator_id=NARRATOR)
next_plan = build_session_plan(
    threads=memory_after.threads,
    sensitivities=memory_after.sensitivities,
    ask=repo.pending_ask(NARRATOR),
    session_count=memory_after.session_count,
    last_closure=memory_after.last_closure,
)
print("greeting now:", next_plan.greeting)
print("offers now  :", [(o.kind.value, o.label) for o in next_plan.offers])
for offer in next_plan.offers:
    print("   would say:", offer.say)

greeting now: ask whether she slept well after last time
offers now  : [('thread', "father's coffee shop"), ('thread', 'childhood in Sungai Siput rubber estate')]
   would say: father's coffee shop — still unfinished: When and how her father started the coffee shop after the rubber estate days
   would say: childhood in Sungai Siput rubber estate — still unfinished: Life in the line houses and estate routine


Extraction feeds the opener; the opener produces conversation; conversationfeeds extraction. That loop is the engine, and it is why *"session 20 has asharper timeline than session 3"* is a mechanism rather than a claim.

## 8. Summary| Component | Where | What it does ||---|---|---|| Domain model | `models.py` | Typed schema; `When`/`Where` keep her words beside the interpretation || Rubric | `models.assess` | Six fields, threshold four; `missing_fields` becomes the next question || Pre-set KB | `seeds/intake.json` | Child-completed family intake; stops painful questions || Call setup | `callflow.prepare_call` | Loads memory, builds the plan, closes tools over it || Session plan | `opener.build_session_plan` | Ranked, sensitivity-gated, depth-gated, max two offers || Instruction | `companion.build_instruction` | Three layers, sent **once** at connect || Pull channel | `tools.recall` | Entity index as lookup key; returns her actual words || Push channel | `tools._with_guidance` | Affect policy rides on every tool response || Affect | `affect.apply_assessment` | Two agreeing readings to move; distress acts immediately || Archivist | `archivist.ingest_conversation` | Seam 1 — text in, structured memory out || Persistence | `repository.py` | Entities, stories, threads, anchors, preferences |### The three seams`ingest_conversation`, `build_session_plan`, and `apply_assessment` are purefunctions with the model calls behind protocols. That is why 297 tests run inunder a second without touching a network.### What I would change1. **Persist the resolutions.** The story→entity edge is computed every ingest   and discarded. Cheapest high-value fix.2. **Stop letting the model own join keys.** Topic labels, entity names and   place names are strings a nondeterministic model produces. It already bit:   one refusal came back as *"the reason the shop closed"* in session 2 and   *"grandfather's shop shutting"* in session 4, so an engagement landed on a   different topic and the subject stayed marked do-not-raise.3. **Model contradiction.** If she contradicts herself in session 9 there is no   principled merge. In an oral history, contradiction is interesting — it   should be a state, not a last-write-wins accident.